In [ ]:
import os

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from neo4j import GraphDatabase

In [119]:
load_dotenv()

NEO4J_URI      = os.getenv("NEO4J_URI")
NEO4J_USER     = os.getenv("NEO4J_USER")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD)
)
driver.verify_connectivity()

def run_read(cypher, **params):
    with driver.session(database=NEO4J_DATABASE) as s:
        return pd.DataFrame([r.data() for r in s.run(cypher, **params)])

In [120]:
query = """
MATCH (s:Struttura) -[:HA_AREA]->(a:AreaSpecialistica)
RETURN s.Nome as struttura, a.Nome as specialita
"""

df = run_read(query)
M  = pd.crosstab(df["struttura"], df["specialita"])
M = (M > 0).astype(int)
M.head(20)

specialita,ALLERGOLOGIA,ANALISI DI LABORATORIO,ANATOMIA PATOLOGICA,ANGIOLOGIA,ASTANTERIA - DEGENZA BREVE,CARDIOCHIRURGIA,CARDIOCHIRURGIA PEDIATRICA,CARDIOLOGIA,CARDIOLOGIA PEDIATRICA,CHECK UP,...,SERVIZI DOMICILIARI,SERVIZI TERRITORIALI,TERAPIA INTENSIVA,TERAPIA INTENSIVA NEONATALE,TERAPIA INTENSIVA PEDIATRICA,TOSSICOLOGIA,UNITÀ CORONARICA - UNITÀ INTENSIVA CARDIOLOGICA,UNITÀ SPINALE,UROLOGIA E ANDROLOGIA,UROLOGIA PEDIATRICA
struttura,,,,,,,,,,,,,,,,,,,,,
AUXOLOGICO ARIOSTO DI MILANO,1,1,0,1,0,1,0,1,0,1,...,1,1,0,0,0,0,0,0,1,0
AUXOLOGICO CAPITANIO DI MILANO,1,1,0,1,0,1,0,1,0,1,...,1,1,0,0,0,0,0,0,1,0
AUXOLOGICO MOSÈ BIANCHI DI MILANO,0,1,0,0,0,1,0,1,0,1,...,1,1,0,0,0,0,0,0,0,0
AUXOLOGICO SAN LUCA DI MILANO,1,1,0,1,0,1,0,1,0,0,...,1,1,0,0,0,0,1,0,1,0
AZIENDA OSPEDALIERA MELLINO MELLINI DI CHIARI,0,1,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,1,0,1,0
CENTRO CARDIOLOGICO MONZINO DI MILANO,0,1,0,1,0,1,0,1,0,0,...,0,0,1,0,0,0,1,0,0,0
EUGENIO MEDEA - POLO SCIENTIFICO DI BOSISIO PARINI,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
FONDAZIONE IRCCS CA' GRANDA OSPEDALE MAGGIORE POLICLINICO DI MILANO,0,0,1,0,1,0,0,1,1,0,...,0,1,1,1,1,0,1,0,1,1
FONDAZIONE IRCCS ISTITUTO NEUROLOGICO CARLO BESTA DI MILANO,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0


In [121]:
diversita = M.sum(axis=1).rename("diversita")
ubiquita  = M.sum(axis=0).rename("ubiquita")

print("Diversità (specialità per struttura):")
print(diversita.describe().round(2))
print("\nUbiquità (strutture per specialità):")
print(ubiquita.describe().round(2))

Diversità (specialità per struttura):
count    134.00
mean      25.96
std       15.42
min        1.00
25%       14.25
50%       25.00
75%       35.00
max       64.00
Name: diversita, dtype: float64

Ubiquità (strutture per specialità):
count     97.00
mean      35.86
std       34.61
min        1.00
25%        4.00
50%       26.00
75%       57.00
max      128.00
Name: ubiquita, dtype: float64


In [122]:
def eci_pci_computation(
    M: pd.DataFrame,
    iter: int = 200,
    ) -> tuple[pd.Series, pd.Series]:

    Mc = M.sum(axis=1).values.astype(float)
    Mp = M.sum(axis=0).values.astype(float)
    Mv = M.values.astype(float)

    denom_c = np.where(Mc == 0, 1.0, Mc)
    denom_p = np.where(Mp == 0, 1.0, Mp)

    Kc = Mc.copy()
    Kc -= Kc.mean()

    for i in range(iter):
        Kp_new = (Mv.T @ Kc) / denom_p
        Kc_new = (Mv   @ Kp_new) / denom_c

        Kc_new -= Kc_new.mean()

        norm = np.linalg.norm(Kc_new)
        if norm > 0:
            Kc_new /= norm

        if np.linalg.norm(Kc_new - Kc) < 1e-10:
            Kc = Kc_new
            break

        Kc = Kc_new
    else:
        print(f"Loop terminato dopo {iter} iterazioni (nessuna convergenza anticipata).")

    Kp = (Mv.T @ Kc) / denom_p

    if np.corrcoef(Kc, Mc)[0, 1] < 0:
        Kc, Kp = -Kc, -Kp

    def zscore(x: np.ndarray) -> np.ndarray:
        std = x.std()
        return (x - x.mean()) / std if std > 0 else np.zeros_like(x)

    eci = pd.Series(zscore(Kc), index=M.index,   name="ECI")
    pci = pd.Series(zscore(Kp), index=M.columns, name="PCI")
    return eci, pci


eci, pci = eci_pci_computation(M)
print(f"\nECI — min: {eci.min():.3f}  max: {eci.max():.3f}  mean: {eci.mean():.3f}")
print(f"PCI — min: {pci.min():.3f}  max: {pci.max():.3f}  mean: {pci.mean():.3f}")


ECI — min: -1.243  max: 10.367  mean: 0.000
PCI — min: -0.582  max: 2.411  mean: 0.000


In [123]:
df_report = (
    pd.DataFrame({"diversita": diversita, "ECI": eci})
    .sort_values("ECI", ascending=False)
)

df_report.head(10)

,diversita,ECI
struttura,,
FONDAZIONE IRCCS CA' GRANDA OSPEDALE MAGGIORE POLICLINICO DI MILANO,63,10.366669
OSPEDALE FILIPPO DEL PONTE DI VARESE,14,1.071163
OSPEDALE POLICLINICO SAN MATTEO DI PAVIA,59,1.007119
OSPEDALE PAPA GIOVANNI XXIII DI BERGAMO,63,0.986333
PRESIDIO OSPEDALIERO GAETANO PINI,23,0.918127
PRESIDIO OSPEDALIERO SPEDALI CIVILI DI BRESCIA,54,0.883739
OSPEDALE MACEDONIO MELLONI DI MILANO,14,0.810980
OSPEDALE DI CIRCOLO E FONDAZIONE MACCHI DI VARESE,55,0.793215
IRCCS OSPEDALE SAN RAFFAELE DI MILANO - GRUPPO SAN DONATO,64,0.768636


In [124]:
pci.sort_values(ascending=False).head(10).to_frame()

,PCI
specialita,
CARDIOLOGIA PEDIATRICA,2.411143
DIALISI E TRAPIANTO PEDIATRICO,2.411143
CHIRURGIA DERMATOLOGICA E DERMATOLOGIA PEDIATRICA,2.411143
CHIRURGIA SENOLOGICA,2.411143
EMATOLOGIA,2.411143
GASTROENTEROLOGIA ED ENDOSCOPIA,2.411143
ODONTOSTOMATOLOGIA,2.411143
NEUROPSICHIATRIA DELL'INFANZIA,2.411143
MEDICINA GENERALE,2.411143


In [125]:
df_report.to_csv("eci_strutture.csv")
pci.sort_values(ascending=False).to_csv("pci_specialita.csv")